In [2]:
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Définition du chemin d'accès aux données
data_path = "/home/kaoutar/Desktop/data"  # Ajustez ce chemin selon votre système

# Tentative de différents encodages courants
encodings = ['latin1', 'ISO-8859-1', 'utf-8', 'cp1252', 'utf-16', 'ascii']

# Fonction pour essayer différents encodages avec plus d'informations de débogage
def read_with_encoding(file_path, separator='|', encodings=encodings):
    if not os.path.exists(file_path):
        print(f"Fichier non trouvé: {file_path}")
        return pd.DataFrame()
        
    for encoding in encodings:
        try:
            df = pd.read_csv(file_path, sep=separator, encoding=encoding, on_bad_lines='skip')
            print(f"Fichier lu avec succès: {file_path}, encodage: {encoding}")
            return df
        except UnicodeDecodeError:
            continue
        except Exception as e:
            print(f"Erreur lors de la lecture de {file_path}: {str(e)}")
            continue
    
    # Si aucun encodage ne fonctionne
    print(f"Impossible de lire le fichier {file_path} avec les encodages essayés")
    return pd.DataFrame()  # Retourner un DataFrame vide

# Chargement des tables avec gestion d'encodage
df_credit = read_with_encoding(os.path.join(data_path, 'ACT_CD_CREDIT_MDM28032025.txt'))
df_getchq = read_with_encoding(os.path.join(data_path, 'ACT_CD_GETCHQ_MDM28032025.txt'))
df_operations = read_with_encoding(os.path.join(data_path, 'ACT_CD_OPERATIONS_AGENCE_MDM20032025.txt'), separator=';')
df_vignette = read_with_encoding(os.path.join(data_path, 'ACT_CD_PAI_VIGNETTE_MDM20032025.txt'))
df_produit = read_with_encoding(os.path.join(data_path, 'ACT_CD_PRODUIT_MDM20032025.txt'))
df_solde = read_with_encoding(os.path.join(data_path, 'ACT_CD_SOLDE_CLIENT_MDM28032025.txt'))
df_conn = read_with_encoding(os.path.join(data_path, 'ACT_CD_STAT_CONNEXION_MDM28032025.txt'))
df_trans = read_with_encoding(os.path.join(data_path, 'ACT_CD_TRANSA_MDM20032025.txt'))
df_transferts = read_with_encoding(os.path.join(data_path, 'ACT_CD_TRANSFERTS_COMPTE_MDMV20032025.txt'))
df_equi = read_with_encoding(os.path.join(data_path, 'ACT_EQUI_MDM28032025.txt'))
df_perimetre = read_with_encoding(os.path.join(data_path, 'Périmétre_projet.txt'))

# Vérification des données chargées
for name, df in zip(['Crédit', 'GetChq', 'Opérations', 'Vignette', 'Produit', 'Solde', 'Connexion', 
                    'Transactions', 'Transferts', 'Équipements', 'Périmètre'],
                   [df_credit, df_getchq, df_operations, df_vignette, df_produit, df_solde, 
                    df_conn, df_trans, df_transferts, df_equi, df_perimetre]):
    print(f"Table {name}: {df.shape[0]} lignes, {df.shape[1]} colonnes")
    if not df.empty:
        print(f"Colonnes: {df.columns.tolist()}")
        # Vérifier la présence de la colonne RADICAL ou CONTRAT
        id_col = None
        if 'RADICAL' in df.columns:
            id_col = 'RADICAL'
        elif 'CONTRAT' in df.columns:
            id_col = 'CONTRAT'
        
        if id_col:
            print(f"Nombre de clients uniques: {df[id_col].nunique()}")
        else:
            print("Pas de colonne d'identification client trouvée")
    else:
        print("Dataframe vide - problème de chargement")
    print("-" * 80)

# Continuer seulement si les dataframes essentiels ne sont pas vides
essential_dfs = [df_conn, df_trans, df_getchq, df_vignette]
essential_names = ['Connexion', 'Transactions', 'GetChq', 'Vignette']

missing_dfs = [name for df, name in zip(essential_dfs, essential_names) if df.empty]
if missing_dfs:
    print(f"Impossible de continuer l'analyse car les dataframes suivants sont vides: {', '.join(missing_dfs)}")
else:
    # Standardisation des noms de colonnes entre tables
    for df in [df_conn, df_getchq, df_trans, df_vignette]:
        if 'CONTRAT' in df.columns and 'RADICAL' not in df.columns:
            df['RADICAL'] = df['CONTRAT']
    
    # Vérification des colonnes nécessaires
    missing_columns = []
    if 'CONNECTION' not in df_conn.columns:
        missing_columns.append("'CONNECTION' dans la table Connexion")
    if 'NOMBRE_OPE' not in df_trans.columns:
        missing_columns.append("'NOMBRE_OPE' dans la table Transactions")
    
    if missing_columns:
        print(f"Colonnes manquantes: {', '.join(missing_columns)}")
        print("Veuillez vérifier les noms des colonnes dans vos fichiers source")
    else:
        # Création des indicateurs pour la variable cible
        # 1. Indicateur de connexion
        connexion_clients = df_conn[df_conn['CONNECTION'] != 'Jamais Connecte']['RADICAL'].unique()
        df_connexion = pd.DataFrame({'RADICAL': connexion_clients, 'is_connected': True})
        
        # 2. Indicateur de transactions (au moins 2)
        transactions_par_client = df_trans.groupby('RADICAL')['NOMBRE_OPE'].sum().reset_index()
        transactions_clients = transactions_par_client[transactions_par_client['NOMBRE_OPE'] >= 2]['RADICAL'].unique()
        df_transactions = pd.DataFrame({'RADICAL': transactions_clients, 'has_transactions': True})
        
        # 3. Indicateur GETCHQ
        getchq_clients = df_getchq['RADICAL'].unique()
        df_getchq_indicator = pd.DataFrame({'RADICAL': getchq_clients, 'used_getchq': True})
        
        # 4. Indicateur vignette
        vignette_clients = df_vignette['RADICAL'].unique()
        df_vignette_indicator = pd.DataFrame({'RADICAL': vignette_clients, 'paid_vignette': True})
        
        # Création du dataframe de base avec tous les clients
        all_clients = set()
        for df in [df_perimetre, df_credit, df_operations, df_solde, df_trans, df_transferts, df_equi, df_vignette, df_conn, df_getchq]:
            if not df.empty and 'RADICAL' in df.columns:
                all_clients.update(df['RADICAL'].unique())
        
        df_main = pd.DataFrame({'RADICAL': list(all_clients)})
        
        # Fusion avec les indicateurs
        df_main = df_main.merge(df_connexion, on='RADICAL', how='left').fillna(False)
        df_main = df_main.merge(df_transactions, on='RADICAL', how='left').fillna(False)
        df_main = df_main.merge(df_getchq_indicator, on='RADICAL', how='left').fillna(False)
        df_main = df_main.merge(df_vignette_indicator, on='RADICAL', how='left').fillna(False)
        
        # Création de la variable cible
        df_main['appetent'] = df_main['is_connected'] & (df_main['has_transactions'] | df_main['used_getchq'] | df_main['paid_vignette'])
        
        # Vérification de la distribution de la variable cible
        print(f"Nombre total de clients: {len(df_main)}")
        print(f"Nombre de clients appétents: {df_main['appetent'].sum()}")
        print(f"Pourcentage de clients appétents: {df_main['appetent'].mean() * 100:.2f}%")
        
        # Fusion avec les données démographiques si disponibles
        if not df_perimetre.empty:
            df_main = df_main.merge(df_perimetre, on='RADICAL', how='left')
            
            # Transformation de la date de naissance en âge si la colonne existe
            if 'DATE_OF_BIRTH' in df_main.columns:
                def calculate_age(birth_date):
                    if pd.isna(birth_date):
                        return np.nan
                    try:
                        birth_date_str = str(birth_date)
                        # Essayer différents formats possibles
                        if len(birth_date_str) >= 4:
                            birth_year = int(birth_date_str[:4])
                            current_year = 2025  # Année actuelle selon vos données
                            age = current_year - birth_year
                            # Vérifier que l'âge est réaliste
                            if 0 <= age <= 120:
                                return age
                        return np.nan
                    except:
                        return np.nan
                
                df_main['AGE'] = df_main['DATE_OF_BIRTH'].apply(calculate_age)
            
            # Calcul de l'ancienneté client (en années) si la colonne existe
            if 'CUSTOMER_SINCE' in df_main.columns:
                def calculate_tenure(customer_since):
                    if pd.isna(customer_since):
                        return np.nan
                    try:
                        start_year = int(str(customer_since)[:4])
                        current_year = 2025  # Année actuelle selon vos données
                        return current_year - start_year
                    except:
                        return np.nan
                
                df_main['TENURE'] = df_main['CUSTOMER_SINCE'].apply(calculate_tenure)
        
        # Enrichissement avec les données financières agrégées
        # Solde moyen
        if not df_solde.empty and 'SOLDE' in df_solde.columns:
            # Convertir SOLDE en numérique si ce n'est pas déjà le cas
            try:
                df_solde['SOLDE'] = pd.to_numeric(df_solde['SOLDE'], errors='coerce')
                solde_moyen = df_solde.groupby('RADICAL')['SOLDE'].mean().reset_index()
                df_main = df_main.merge(solde_moyen, on='RADICAL', how='left')
                df_main.rename(columns={'SOLDE': 'SOLDE_MOYEN'}, inplace=True)
            except Exception as e:
                print(f"Erreur lors du calcul du solde moyen: {str(e)}")
        
        # Nombre moyen d'opérations mensuelles
        if not df_operations.empty and 'NOMBRE_OPERATION' in df_operations.columns:
            try:
                df_operations['NOMBRE_OPERATION'] = pd.to_numeric(df_operations['NOMBRE_OPERATION'], errors='coerce')
                operations_mensuelles = df_operations.groupby('RADICAL')['NOMBRE_OPERATION'].mean().reset_index()
                df_main = df_main.merge(operations_mensuelles, on='RADICAL', how='left')
                df_main.rename(columns={'NOMBRE_OPERATION': 'OPERATIONS_MENSUELLES_MOY'}, inplace=True)
            except Exception as e:
                print(f"Erreur lors du calcul des opérations mensuelles: {str(e)}")
        
        # Indicateur de possession de crédit
        if not df_credit.empty:
            df_main['HAS_CREDIT'] = df_main['RADICAL'].isin(df_credit['RADICAL'])
        
        # Indicateur de produit packagé (BLADI)
        if not df_produit.empty and 'LIB_PRD' in df_produit.columns:
            df_main['HAS_BLADI'] = df_main['RADICAL'].isin(df_produit[df_produit['LIB_PRD'] == 'BLADI']['RADICAL'])
        
        # Sauvegarder le dataframe principal pour utilisation ultérieure
        try:
            df_main.to_csv(os.path.join(data_path, 'df_main_prepared.csv'), index=False)
            print(f"Fichier sauvegardé: {os.path.join(data_path, 'df_main_prepared.csv')}")
        except Exception as e:
            print(f"Erreur lors de la sauvegarde du fichier principal: {str(e)}")
        
        # Analyse exploratoire initiale
        if df_main.shape[0] > 0:
            try:
                # Distribution des clients appétents vs non-appétents
                plt.figure(figsize=(10, 6))
                sns.countplot(x='appetent', data=df_main)
                plt.title('Distribution des clients appétents vs non-appétents')
                plt.xlabel('Client appétent')
                plt.ylabel('Nombre de clients')
                plt.xticks([0, 1], ['Non appétent', 'Appétent'])
                plt.savefig(os.path.join(data_path, 'distribution_appetence.png'))
                plt.close()
                print(f"Graphique sauvegardé: distribution_appetence.png")
            except Exception as e:
                print(f"Erreur lors de la création du graphique distribution_appetence: {str(e)}")
            
            try:
                # Distribution par profil client si la colonne existe
                if 'CR_PROFIL' in df_main.columns:
                    plt.figure(figsize=(12, 6))
                    sns.countplot(x='CR_PROFIL', hue='appetent', data=df_main)
                    plt.title('Appétence digitale par profil client')
                    plt.xlabel('Profil client')
                    plt.ylabel('Nombre de clients')
                    plt.xticks(rotation=45)
                    plt.legend(['Non appétent', 'Appétent'])
                    plt.tight_layout()
                    plt.savefig(os.path.join(data_path, 'appetence_par_profil.png'))
                    plt.close()
                    print(f"Graphique sauvegardé: appetence_par_profil.png")
            except Exception as e:
                print(f"Erreur lors de la création du graphique appetence_par_profil: {str(e)}")
            
            try:
                # Distribution par pays de résidence si la colonne existe
                if 'RESIDENCE' in df_main.columns:
                    plt.figure(figsize=(14, 6))
                    # Cette section a été modifiée pour éviter l'erreur
                    # Au lieu de faire un groupby avec mean, on fait un comptage explicite
                    cross_tab = pd.crosstab(df_main['RESIDENCE'], df_main['appetent'])
                    
                    # Calculer le pourcentage d'appétents pour chaque pays
                    if True in cross_tab.columns:  # S'assurer que la colonne True existe
                        total = cross_tab.sum(axis=1)
                        pct_appetent = (cross_tab[True] / total * 100).sort_values(ascending=False)
                        
                        # Limiter aux 15 premiers pays pour la lisibilité
                        pct_appetent.head(15).plot(kind='bar')
                        plt.title('Pourcentage de clients appétents par pays de résidence')
                        plt.xlabel('Pays de résidence')
                        plt.ylabel('Pourcentage de clients appétents')
                        plt.tight_layout()
                        plt.savefig(os.path.join(data_path, 'appetence_par_residence.png'))
                        print(f"Graphique sauvegardé: appetence_par_residence.png")
                    else:
                        print("Pas de clients appétents trouvés pour l'analyse par résidence")
                    plt.close()
            except Exception as e:
                print(f"Erreur lors de la création du graphique appetence_par_residence: {str(e)}")
            
            try:
                # Distribution par tranche d'âge si la colonne âge existe
                if 'AGE' in df_main.columns:
                    # S'assurer que AGE est numérique
                    df_main['AGE'] = pd.to_numeric(df_main['AGE'], errors='coerce')
                    
                    # Créer des tranches d'âge seulement si les données sont valides
                    if df_main['AGE'].notna().sum() > 10:  # Au moins 10 valeurs non-NA
                        df_main['AGE_GROUP'] = pd.cut(df_main['AGE'], 
                                                   bins=[0, 30, 40, 50, 60, 100], 
                                                   labels=['<30', '30-40', '40-50', '50-60', '>60'],
                                                   include_lowest=True)
                        
                        plt.figure(figsize=(12, 6))
                        sns.countplot(x='AGE_GROUP', hue='appetent', data=df_main.dropna(subset=['AGE_GROUP']))
                        plt.title('Appétence digitale par tranche d\'âge')
                        plt.xlabel('Tranche d\'âge')
                        plt.ylabel('Nombre de clients')
                        plt.legend(['Non appétent', 'Appétent'])
                        plt.tight_layout()
                        plt.savefig(os.path.join(data_path, 'appetence_par_age.png'))
                        plt.close()
                        print(f"Graphique sauvegardé: appetence_par_age.png")
                    else:
                        print("Pas assez de données d'âge valides pour l'analyse")
            except Exception as e:
                print(f"Erreur lors de la création du graphique appetence_par_age: {str(e)}")
            
            try:
                # Corrélation entre variables numériques et appétence
                numeric_cols = ['AGE', 'TENURE', 'SOLDE_MOYEN', 'OPERATIONS_MENSUELLES_MOY']
                # Filtrer seulement les colonnes qui existent
                existing_cols = [col for col in numeric_cols if col in df_main.columns]
                
                if existing_cols:
                    # S'assurer que toutes les colonnes sont numériques
                    numeric_df = df_main[existing_cols + ['appetent']].copy()
                    
                    for col in existing_cols:
                        numeric_df[col] = pd.to_numeric(numeric_df[col], errors='coerce')
                    
                    # Convertir appetent en numérique
                    numeric_df['appetent'] = numeric_df['appetent'].astype(int)
                    
                    # Supprimer les lignes avec des valeurs manquantes
                    numeric_df = numeric_df.dropna()
                    
                    if not numeric_df.empty and numeric_df.shape[0] > 10:
                        correlation_data = numeric_df.corr()['appetent'].sort_values(ascending=False)
                        
                        plt.figure(figsize=(10, 6))
                        correlation_data.drop('appetent').plot(kind='bar')
                        plt.title('Corrélation entre variables numériques et appétence digitale')
                        plt.xlabel('Variables')
                        plt.ylabel('Coefficient de corrélation')
                        plt.tight_layout()
                        plt.savefig(os.path.join(data_path, 'correlation_appetence.png'))
                        plt.close()
                        print(f"Graphique sauvegardé: correlation_appetence.png")
                    else:
                        print("Pas assez de données numériques pour l'analyse de corrélation")
            except Exception as e:
                print(f"Erreur lors de la création du graphique correlation_appetence: {str(e)}")
        
        print("Analyse terminée. Les fichiers ont été sauvegardés dans le répertoire:", data_path)

Fichier lu avec succès: /home/kaoutar/Desktop/data/ACT_CD_CREDIT_MDM28032025.txt, encodage: latin1
Fichier lu avec succès: /home/kaoutar/Desktop/data/ACT_CD_GETCHQ_MDM28032025.txt, encodage: latin1
Fichier lu avec succès: /home/kaoutar/Desktop/data/ACT_CD_OPERATIONS_AGENCE_MDM20032025.txt, encodage: latin1
Fichier lu avec succès: /home/kaoutar/Desktop/data/ACT_CD_PAI_VIGNETTE_MDM20032025.txt, encodage: latin1


/tmp/ipykernel_6851/1649940419.py:22: DtypeWarning: Columns (3,5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, sep=separator, encoding=encoding, on_bad_lines='skip')


Fichier lu avec succès: /home/kaoutar/Desktop/data/ACT_CD_PRODUIT_MDM20032025.txt, encodage: latin1
Fichier lu avec succès: /home/kaoutar/Desktop/data/ACT_CD_SOLDE_CLIENT_MDM28032025.txt, encodage: latin1
Fichier lu avec succès: /home/kaoutar/Desktop/data/ACT_CD_STAT_CONNEXION_MDM28032025.txt, encodage: latin1
Fichier lu avec succès: /home/kaoutar/Desktop/data/ACT_CD_TRANSA_MDM20032025.txt, encodage: latin1
Fichier lu avec succès: /home/kaoutar/Desktop/data/ACT_CD_TRANSFERTS_COMPTE_MDMV20032025.txt, encodage: latin1
Fichier lu avec succès: /home/kaoutar/Desktop/data/ACT_EQUI_MDM28032025.txt, encodage: latin1
Fichier lu avec succès: /home/kaoutar/Desktop/data/Périmétre_projet.txt, encodage: latin1
Table Crédit: 32835 lignes, 11 colonnes
Colonnes: ['RADICAL', 'NODOSS', 'CD_MTIF_DOSS', 'DATE_OCTROI', 'MONTANT', 'DUREE', 'DT_FIN_DOSS_REEL', 'CAPITAL_RES', 'TYPE', 'SOURCE', 'IMP']
Nombre de clients uniques: 31138
------------------------------------------------------------------------------

/tmp/ipykernel_6851/1649940419.py:122: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_main = df_main.merge(df_connexion, on='RADICAL', how='left').fillna(False)
/tmp/ipykernel_6851/1649940419.py:123: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_main = df_main.merge(df_transactions, on='RADICAL', how='left').fillna(False)
/tmp/ipykernel_6851/1649940419.py:124: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the fut

Nombre total de clients: 1279226
Nombre de clients appétents: 69715
Pourcentage de clients appétents: 5.45%
Fichier sauvegardé: /home/kaoutar/Desktop/data/df_main_prepared.csv
Graphique sauvegardé: distribution_appetence.png
Graphique sauvegardé: appetence_par_profil.png
Graphique sauvegardé: appetence_par_residence.png
Graphique sauvegardé: appetence_par_age.png
Graphique sauvegardé: correlation_appetence.png
Analyse terminée. Les fichiers ont été sauvegardés dans le répertoire: /home/kaoutar/Desktop/data


In [3]:
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_classif, SelectKBest
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE
from scipy import stats
import warnings

# Ignore warnings for cleaner output
warnings.filterwarnings('ignore')

# Définition du chemin d'accès aux données
data_path = "/home/kaoutar/Desktop/data"  # Ajustez ce chemin selon votre système

# Si le fichier préparé existe déjà, le charger directement
prepared_file = os.path.join(data_path, 'df_main_prepared.csv')
if os.path.exists(prepared_file):
    print(f"Chargement du fichier préparé existant: {prepared_file}")
    df_main = pd.read_csv(prepared_file)
    print(f"Données chargées: {df_main.shape[0]} lignes, {df_main.shape[1]} colonnes")
else:
    # Code de préparation initial (comme dans le script précédent)
    # Tentative de différents encodages courants
    encodings = ['latin1', 'ISO-8859-1', 'utf-8', 'cp1252', 'utf-16', 'ascii']
    
    # Fonction pour essayer différents encodages avec plus d'informations de débogage
    def read_with_encoding(file_path, separator='|', encodings=encodings):
        if not os.path.exists(file_path):
            print(f"Fichier non trouvé: {file_path}")
            return pd.DataFrame()
            
        for encoding in encodings:
            try:
                df = pd.read_csv(file_path, sep=separator, encoding=encoding, on_bad_lines='skip')
                print(f"Fichier lu avec succès: {file_path}, encodage: {encoding}")
                return df
            except UnicodeDecodeError:
                continue
            except Exception as e:
                print(f"Erreur lors de la lecture de {file_path}: {str(e)}")
                continue
        
        # Si aucun encodage ne fonctionne
        print(f"Impossible de lire le fichier {file_path} avec les encodages essayés")
        return pd.DataFrame()  # Retourner un DataFrame vide
    
    # Chargement des tables avec gestion d'encodage
    df_credit = read_with_encoding(os.path.join(data_path, 'ACT_CD_CREDIT_MDM28032025.txt'))
    df_getchq = read_with_encoding(os.path.join(data_path, 'ACT_CD_GETCHQ_MDM28032025.txt'))
    df_operations = read_with_encoding(os.path.join(data_path, 'ACT_CD_OPERATIONS_AGENCE_MDM20032025.txt'), separator=';')
    df_vignette = read_with_encoding(os.path.join(data_path, 'ACT_CD_PAI_VIGNETTE_MDM20032025.txt'))
    df_produit = read_with_encoding(os.path.join(data_path, 'ACT_CD_PRODUIT_MDM20032025.txt'))
    df_solde = read_with_encoding(os.path.join(data_path, 'ACT_CD_SOLDE_CLIENT_MDM28032025.txt'))
    df_conn = read_with_encoding(os.path.join(data_path, 'ACT_CD_STAT_CONNEXION_MDM28032025.txt'))
    df_trans = read_with_encoding(os.path.join(data_path, 'ACT_CD_TRANSA_MDM20032025.txt'))
    df_transferts = read_with_encoding(os.path.join(data_path, 'ACT_CD_TRANSFERTS_COMPTE_MDMV20032025.txt'))
    df_equi = read_with_encoding(os.path.join(data_path, 'ACT_EQUI_MDM28032025.txt'))
    df_perimetre = read_with_encoding(os.path.join(data_path, 'Périmétre_projet.txt'))
    
    # Continuer avec le code de préparation comme avant...
    # (Je suppose que vous avez déjà préparé le fichier, donc j'exclus cette partie)
    print("Veuillez exécuter le script initial pour préparer les données d'abord.")
    exit()

# Créer un dossier pour les visualisations
viz_path = os.path.join(data_path, 'visualisations')
if not os.path.exists(viz_path):
    os.makedirs(viz_path)

print("Début de l'analyse exploratoire approfondie...")

# 1. FEATURE ENGINEERING & EXPLORATION
print("\n" + "="*50)
print("1. FEATURE ENGINEERING & EXPLORATION")
print("="*50)

# Aperçu initial des données
print("\nAperçu des données:")
print(df_main.head())

# Information sur les colonnes
print("\nInformations sur les colonnes:")
df_info = pd.DataFrame({
    'Type': df_main.dtypes,
    'Non-Null Count': df_main.count(),
    'Null Count': df_main.isnull().sum(),
    'Null Percentage': df_main.isnull().mean() * 100,
    'Unique Values': df_main.nunique()
})
print(df_info)

# Créer de nouvelles features si possible
print("\nCréation de nouvelles features...")

# Identifier les colonnes numériques et catégorielles
numeric_cols = df_main.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = df_main.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()



# Exclure 'RADICAL' et 'appetent' de ces listes
if 'RADICAL' in numeric_cols:
    numeric_cols.remove('RADICAL')
if 'appetent' in numeric_cols:
    numeric_cols.remove('appetent')
if 'RADICAL' in categorical_cols:
    categorical_cols.remove('RADICAL')
if 'appetent' in categorical_cols:
    categorical_cols.remove('appetent')

print(f"\nColonnes numériques identifiées: {numeric_cols}")
print(f"Colonnes catégorielles identifiées: {categorical_cols}")

# Créer des ratio et features dérivées si possible
if 'SOLDE_MOYEN' in df_main.columns and 'OPERATIONS_MENSUELLES_MOY' in df_main.columns:
    # S'assurer que les colonnes sont numériques
    df_main['SOLDE_MOYEN'] = pd.to_numeric(df_main['SOLDE_MOYEN'], errors='coerce')
    df_main['OPERATIONS_MENSUELLES_MOY'] = pd.to_numeric(df_main['OPERATIONS_MENSUELLES_MOY'], errors='coerce')
    
    # Créer un ratio d'activité (opérations par unité de solde)
    df_main['RATIO_ACTIVITE'] = df_main['OPERATIONS_MENSUELLES_MOY'] / (df_main['SOLDE_MOYEN'].abs() + 1)  # +1 pour éviter division par zéro
    numeric_cols.append('RATIO_ACTIVITE')
    
    print("Feature créée: RATIO_ACTIVITE (opérations par unité de solde)")

# Si on a l'âge et l'ancienneté, on peut créer des features d'âge lors de l'ouverture du compte
if 'AGE' in df_main.columns and 'TENURE' in df_main.columns:
    df_main['AGE'] = pd.to_numeric(df_main['AGE'], errors='coerce')
    df_main['TENURE'] = pd.to_numeric(df_main['TENURE'], errors='coerce')
    
    df_main['AGE_A_OUVERTURE'] = df_main['AGE'] - df_main['TENURE']
    numeric_cols.append('AGE_A_OUVERTURE')
    
    # Créer des groupes d'âge et d'ancienneté
    df_main['AGE_GROUP'] = pd.cut(df_main['AGE'], 
                                bins=[0, 30, 40, 50, 60, 100], 
                                labels=['<30', '30-40', '40-50', '50-60', '>60'],
                                include_lowest=True)
    
    df_main['TENURE_GROUP'] = pd.cut(df_main['TENURE'], 
                                  bins=[0, 1, 3, 5, 10, 100], 
                                  labels=['<1 an', '1-3 ans', '3-5 ans', '5-10 ans', '>10 ans'],
                                  include_lowest=True)
    
    categorical_cols.extend(['AGE_GROUP', 'TENURE_GROUP'])
    
    print("Features créées: AGE_A_OUVERTURE, AGE_GROUP, TENURE_GROUP")

# 2. ADVANCED STATISTICAL ANALYSIS
print("\n" + "="*50)
print("2. ADVANCED STATISTICAL ANALYSIS")
print("="*50)

# 2.1 Analyse des valeurs manquantes
print("\nAnalyse des valeurs manquantes:")
missing_data = df_main.isnull().sum().sort_values(ascending=False)
missing_percent = (df_main.isnull().sum() / df_main.isnull().count()).sort_values(ascending=False)
missing_data_percent = pd.concat([missing_data, missing_percent], axis=1, 
                                keys=['Total', 'Percent'])
print(missing_data_percent[missing_data_percent['Total'] > 0])

# Visualisation des valeurs manquantes
plt.figure(figsize=(12, 6))
plt.title('Pourcentage de valeurs manquantes par colonne')
missing_percent = missing_percent[missing_percent > 0]
missing_percent.sort_values().plot(kind='barh')
plt.xlabel('Pourcentage')
plt.tight_layout()
plt.savefig(os.path.join(viz_path, '01_missing_values.png'))
plt.close()

# 2.2 Analyse des outliers pour les variables numériques
print("\nAnalyse des outliers pour les variables numériques:")
for col in numeric_cols:
    if col in df_main.columns:
        try:
            # Convertir en numérique au cas où
            df_main[col] = pd.to_numeric(df_main[col], errors='coerce')
            
            # Calculer les statistiques
            q1 = df_main[col].quantile(0.25)
            q3 = df_main[col].quantile(0.75)
            iqr = q3 - q1
            lower_bound = q1 - 1.5 * iqr
            upper_bound = q3 + 1.5 * iqr
            
            # Compter les outliers
            outliers = df_main[(df_main[col] < lower_bound) | (df_main[col] > upper_bound)].shape[0]
            pct_outliers = outliers / df_main.shape[0] * 100
            
            print(f"{col}: {outliers} outliers ({pct_outliers:.2f}%)")
            
            # Boxplot pour visualiser les outliers
            plt.figure(figsize=(10, 6))
            sns.boxplot(x=df_main[col])
            plt.title(f'Distribution et outliers pour {col}')
            plt.tight_layout()
            plt.savefig(os.path.join(viz_path, f'02_outliers_{col}.png'))
            plt.close()
        except Exception as e:
            print(f"  Erreur lors de l'analyse des outliers pour {col}: {str(e)}")

# 2.3 Tests statistiques
print("\nTests statistiques pour les variables catégorielles (Chi-square):")
for col in categorical_cols:
    if col in df_main.columns and df_main[col].nunique() < 50:  # Limiter aux variables avec moins de 50 catégories
        try:
            # Tableau de contingence
            contingency_table = pd.crosstab(df_main[col], df_main['appetent'])
            
            # Test du chi2
            chi2, p, dof, expected = stats.chi2_contingency(contingency_table)
            
            if p < 0.05:
                significance = "SIGNIFICATIF"
            else:
                significance = "Non significatif"
                
            print(f"{col}: Chi2={chi2:.2f}, p-value={p:.4f} - {significance}")
            
            # Visualisation du tableau de contingence (pourcentage d'appétents par catégorie)
            if contingency_table.shape[0] <= 15:  # Limiter aux variables avec moins de 15 catégories pour la lisibilité
                plt.figure(figsize=(12, 6))
                # Calculer le pourcentage d'appétents par catégorie
                if True in contingency_table.columns:
                    pct_appetent = contingency_table[True] / contingency_table.sum(axis=1) * 100
                    pct_appetent.sort_values(ascending=False).plot(kind='bar')
                    plt.title(f'Pourcentage de clients appétents par {col}')
                    plt.xlabel(col)
                    plt.ylabel('Pourcentage')
                    plt.tight_layout()
                    plt.savefig(os.path.join(viz_path, f'03_chi2_{col}.png'))
                plt.close()
        except Exception as e:
            print(f"  Erreur lors du test chi2 pour {col}: {str(e)}")

print("\nTests statistiques pour les variables numériques (ANOVA):")
for col in numeric_cols:
    if col in df_main.columns:
        try:
            # Convertir en numérique et supprimer les NaN
            df_main[col] = pd.to_numeric(df_main[col], errors='coerce')
            temp_df = df_main[[col, 'appetent']].dropna()
            
            if temp_df.shape[0] > 10:  # S'assurer qu'il y a suffisamment de données
                # Séparer les groupes
                appetent_group = temp_df[temp_df['appetent'] == True][col]
                non_appetent_group = temp_df[temp_df['appetent'] == False][col]
                
                if len(appetent_group) > 0 and len(non_appetent_group) > 0:
                    # Test ANOVA
                    f_stat, p_value = stats.f_oneway(appetent_group, non_appetent_group)
                    
                    if p_value < 0.05:
                        significance = "SIGNIFICATIF"
                    else:
                        significance = "Non significatif"
                        
                    print(f"{col}: F={f_stat:.2f}, p-value={p_value:.4f} - {significance}")
                    
                    # Visualisation de la distribution par groupe
                    plt.figure(figsize=(10, 6))
                    sns.histplot(data=temp_df, x=col, hue='appetent', kde=True, common_norm=False)
                    plt.title(f'Distribution de {col} par groupe d\'appétence')
                    plt.tight_layout()
                    plt.savefig(os.path.join(viz_path, f'04_anova_{col}.png'))
                    plt.close()
        except Exception as e:
            print(f"  Erreur lors du test ANOVA pour {col}: {str(e)}")

# 2.4 Feature importance avec mutual information
print("\nFeature importance avec mutual information:")
features_for_mi = []
feature_names = []

# Préparer les données pour le calcul de mutual information
df_for_mi = df_main.copy()

# Traiter les colonnes numériques
for col in numeric_cols:
    if col in df_for_mi.columns:
        df_for_mi[col] = pd.to_numeric(df_for_mi[col], errors='coerce')
        # Imputer les valeurs manquantes avec la médiane
        df_for_mi[col] = df_for_mi[col].fillna(df_for_mi[col].median())
        features_for_mi.append(df_for_mi[col])
        feature_names.append(col)

# Traiter les colonnes catégorielles
for col in categorical_cols:
    if col in df_for_mi.columns and df_for_mi[col].nunique() < 50:  # Limiter aux catégories raisonnables
        # One-hot encoding simplifié pour les catégories
        dummies = pd.get_dummies(df_for_mi[col], prefix=col, drop_first=True)
        for dummy_col in dummies.columns:
            features_for_mi.append(dummies[dummy_col])
            feature_names.append(dummy_col)

if features_for_mi and 'appetent' in df_for_mi.columns:
    try:
        # Convertir en matrice numpy
        X = np.column_stack(features_for_mi)
        y = df_for_mi['appetent'].astype(int)
        
        # Calculer le mutual information score
        mi_scores = mutual_info_classif(X, y, random_state=42)
        
        # Créer un DataFrame pour afficher les résultats
        mi_df = pd.DataFrame({'Feature': feature_names, 'MI Score': mi_scores})
        mi_df = mi_df.sort_values(by='MI Score', ascending=False)
        
        print(mi_df.head(15))  # Afficher les 15 meilleures features
        
        # Visualiser les scores
        plt.figure(figsize=(12, 8))
        mi_df.head(15).plot(x='Feature', y='MI Score', kind='barh')
        plt.title('Mutual Information Scores (Top 15 Features)')
        plt.xlabel('Mutual Information Score')
        plt.tight_layout()
        plt.savefig(os.path.join(viz_path, '05_mutual_information.png'))
        plt.close()
    except Exception as e:
        print(f"Erreur lors du calcul du mutual information: {str(e)}")
else:
    print("Pas assez de données pour le calcul du mutual information")

# 3. ADDITIONAL VISUALIZATIONS
print("\n" + "="*50)
print("3. ADDITIONAL VISUALIZATIONS")
print("="*50)

# 3.1 Matrice de corrélation entre variables numériques
print("\nMatrice de corrélation entre variables numériques:")
try:
    # Sélectionner uniquement les colonnes numériques et y ajouter la variable cible
    numeric_df = df_main[numeric_cols + ['appetent']].copy()
    
    # Convertir toutes les colonnes en numérique
    for col in numeric_df.columns:
        if col == 'appetent':
            numeric_df[col] = numeric_df[col].astype(int)
        else:
            numeric_df[col] = pd.to_numeric(numeric_df[col], errors='coerce')
    
    # Calculer la corrélation
    corr_matrix = numeric_df.corr()
    
    # Visualiser la matrice de corrélation
    plt.figure(figsize=(12, 10))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap='coolwarm', square=True, cbar_kws={"shrink": .8})
    plt.title('Matrice de corrélation entre variables numériques')
    plt.tight_layout()
    plt.savefig(os.path.join(viz_path, '06_correlation_matrix.png'))
    plt.close()
    
    # Afficher les corrélations avec la variable cible
    print("Corrélations avec la variable cible 'appetent':")
    print(corr_matrix['appetent'].sort_values(ascending=False))
except Exception as e:
    print(f"Erreur lors de la création de la matrice de corrélation: {str(e)}")

# 3.2 Pairplot pour visualiser les relations entre variables numériques clés
print("\nCréation d'un pairplot pour les variables numériques clés:")
try:
    # Sélectionner les 4-5 variables numériques les plus importantes (basées sur MI ou corrélation)
    top_numeric = []
    
    if 'mi_df' in locals() and not mi_df.empty:
        # Utiliser les scores de mutual information si disponibles
        for feature in mi_df['Feature'].head(10):  # Regarder parmi les 10 meilleures
            base_feature = feature.split('_')[0]  # Récupérer le nom de base (avant le one-hot encoding)
            if base_feature in numeric_cols and base_feature not in top_numeric:
                top_numeric.append(base_feature)
                if len(top_numeric) >= 4:
                    break
    
    if len(top_numeric) < 4 and 'corr_matrix' in locals():
        # Compléter avec les meilleures corrélations si nécessaire
        appetent_corr = corr_matrix['appetent'].abs().sort_values(ascending=False)
        for feature in appetent_corr.index:
            if feature != 'appetent' and feature in numeric_cols and feature not in top_numeric:
                top_numeric.append(feature)
                if len(top_numeric) >= 4:
                    break
    
    # Si toujours pas assez, ajouter les premières variables numériques disponibles
    if len(top_numeric) < 4:
        for col in numeric_cols:
            if col not in top_numeric:
                top_numeric.append(col)
                if len(top_numeric) >= 4:
                    break
    
    if top_numeric:
        print(f"Variables sélectionnées pour le pairplot: {top_numeric}")
        pair_df = df_main[top_numeric + ['appetent']].copy()
        
        # Convertir en numérique et gérer les NaN
        for col in top_numeric:
            pair_df[col] = pd.to_numeric(pair_df[col], errors='coerce')
        
        # Échantillonner si le dataset est trop grand
        if pair_df.shape[0] > 5000:
            pair_df = pair_df.sample(5000, random_state=42)
        
        # Créer le pairplot
        plt.figure(figsize=(12, 10))
        g = sns.pairplot(pair_df, hue='appetent', diag_kind='kde', plot_kws={'alpha': 0.6})
        g.fig.suptitle('Relations entre variables numériques clés', y=1.02)
        plt.tight_layout()
        plt.savefig(os.path.join(viz_path, '07_pairplot.png'))
        plt.close()
    else:
        print("Pas assez de variables numériques pour créer un pairplot")
except Exception as e:
    print(f"Erreur lors de la création du pairplot: {str(e)}")

# 3.3 Visualisation de l'arbre de décision pour comprendre les règles de segmentation
print("\nVisualisation d'un arbre de décision pour comprendre les facteurs d'appétence:")
try:
    # Préparer les données pour l'arbre de décision
    X_tree = []
    feature_names_tree = []
    
    # Sélectionner les features numériques les plus importantes
    numeric_features = top_numeric if 'top_numeric' in locals() and top_numeric else numeric_cols[:4]
    
    # Préparer les données
    for col in numeric_features:
        if col in df_main.columns:
            X_tree.append(pd.to_numeric(df_main[col], errors='coerce').fillna(0))
            feature_names_tree.append(col)
    
    if X_tree:
        X_tree = np.column_stack(X_tree)
        y_tree = df_main['appetent'].astype(int)
        
        # Entraîner un arbre de décision simple
        tree_model = DecisionTreeClassifier(max_depth=3, random_state=42)
        tree_model.fit(X_tree, y_tree)
        
        # Visualiser l'arbre
        plt.figure(figsize=(15, 10))
        plot_tree(tree_model, feature_names=feature_names_tree, class_names=['Non Appetent', 'Appetent'], 
                 filled=True, rounded=True, fontsize=10)
        plt.title("Arbre de décision pour l'appétence digitale")
        plt.tight_layout()
        plt.savefig(os.path.join(viz_path, '08_decision_tree.png'))
        plt.close()
        
        # Afficher l'importance des features
        feature_importance = pd.DataFrame({
            'Feature': feature_names_tree,
            'Importance': tree_model.feature_importances_
        }).sort_values(by='Importance', ascending=False)
        
        print("Importance des features dans l'arbre de décision:")
        print(feature_importance)
        
        # Visualiser l'importance des features
        plt.figure(figsize=(10, 6))
        feature_importance.plot(x='Feature', y='Importance', kind='bar')
        plt.title("Importance des features dans l'arbre de décision")
        plt.tight_layout()
        plt.savefig(os.path.join(viz_path, '09_tree_feature_importance.png'))
        plt.close()
    else:
        print("Pas assez de variables numériques pour créer un arbre de décision")
except Exception as e:
    print(f"Erreur lors de la création de l'arbre de décision: {str(e)}")

# 4. FEATURE SELECTION
print("\n" + "="*50)
print("4. FEATURE SELECTION")
print("="*50)

# 4.1 Sélection de features avec RFE (Recursive Feature Elimination)
print("\nSélection de features avec RFE:")
try:
    # Préparer les données
    X_rfe = []
    feature_names_rfe = []
    
    # Inclure toutes les variables numériques
    for col in numeric_cols:
        if col in df_main.columns:
            X_rfe.append(pd.to_numeric(df_main[col], errors='coerce').fillna(0))
            feature_names_rfe.append(col)
    
    if X_rfe:
        X_rfe = np.column_stack(X_rfe)
        y_rfe = df_main['appetent'].astype(int)
        
        # Initialiser le modèle (LogisticRegression est souvent utilisé pour RFE)
        estimator = LogisticRegression(max_iter=1000, random_state=42)
        
        # Nombre de features à sélectionner (la moitié, ou au moins 3)
        n_features_to_select = max(3, len(feature_names_rfe) // 2)
        
        # Initialiser RFE
        selector = RFE(estimator, n_features_to_select=n_features_to_select, step=1)
        
        # Ajuster le sélecteur
        selector = selector.fit(X_rfe, y_rfe)
        
        # Récupérer les résultats
        rfe_results = pd.DataFrame({
            'Feature': feature_names_rfe,
            'Selected': selector.support_,
            'Ranking': selector.ranking_
        }).sort_values(by='Ranking')
        
        print("Résultats de la sélection RFE:")
        print(rfe_results)
        
        # Visualiser les features sélectionnées
        plt.figure(figsize=(12, 6))
        sns.barplot(x='Feature', y='Ranking', data=rfe_results.sort_values('Ranking'))
        plt.title('Classement des features par RFE (plus petit = meilleur)')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.savefig(os.path.join(viz_path, '10_rfe_ranking.png'))
        plt.close()
    else:
        print("Pas assez de variables numériques pour effectuer une sélection RFE")
except Exception as e:
    print(f"Erreur lors de la sélection RFE: {str(e)}")



Chargement du fichier préparé existant: /home/kaoutar/Desktop/data/df_main_prepared.csv
Données chargées: 1279444 lignes, 19 colonnes
Début de l'analyse exploratoire approfondie...

1. FEATURE ENGINEERING & EXPLORATION

Aperçu des données:
   RADICAL  is_connected  has_transactions  used_getchq  paid_vignette  \
0  4194306          True              True        False          False   
1  4194309         False             False        False          False   
2        6         False             False        False          False   
3        7         False             False        False          False   
4       11         False             False        False          False   

   appetent     GENRE STATUT_MARITAL  DATE_OF_BIRTH CTRY_OF_BIRTH RESIDENCE  \
0      True  Masculin     MariÃ© (e)     19630506.0            CA        CA   
1     False  Masculin     MariÃ© (e)     19640101.0            MA        BE   
2     False  Masculin     MariÃ© (e)     19500101.0            MA        DE   

<Figure size 1200x800 with 0 Axes>

<Figure size 1200x1000 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

In [4]:
# 2.2 Analyse des outliers pour les variables numériques
print("\nAnalyse des outliers pour les variables numériques:")

# Déterminer le nombre de lignes et colonnes pour le subplot
numeric_cols_available = [col for col in numeric_cols if col in df_main.columns]
n_cols = min(3, len(numeric_cols_available))  # Max 3 colonnes
n_rows = (len(numeric_cols_available) + n_cols - 1) // n_cols  # Calcul du nombre de lignes nécessaires

# Créer une figure avec des subplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
axes = axes.flatten() if n_rows * n_cols > 1 else [axes]  # Aplatir pour faciliter l'indexation

# Dictionnaire pour stocker les informations sur les outliers
outlier_stats = {}

# Boucle à travers les variables numériques
for i, col in enumerate(numeric_cols_available):
    try:
        # Convertir en numérique au cas où
        df_main[col] = pd.to_numeric(df_main[col], errors='coerce')
        
        # Calculer les statistiques
        q1 = df_main[col].quantile(0.25)
        q3 = df_main[col].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        
        # Compter les outliers
        outliers = df_main[(df_main[col] < lower_bound) | (df_main[col] > upper_bound)].shape[0]
        pct_outliers = outliers / df_main.shape[0] * 100
        print(f"{col}: {outliers} outliers ({pct_outliers:.2f}%)")
        
        # Stocker les statistiques
        outlier_stats[col] = {'count': outliers, 'percentage': pct_outliers}
        
        # Créer le boxplot sur le subplot correspondant
        sns.boxplot(x=df_main[col], ax=axes[i])
        axes[i].set_title(f'{col}\n{outliers} outliers ({pct_outliers:.2f}%)')
        axes[i].tick_params(axis='x', rotation=45)  # Rotation des étiquettes si nécessaire
        
    except Exception as e:
        print(f" Erreur lors de l'analyse des outliers pour {col}: {str(e)}")
        if i < len(axes):
            axes[i].set_visible(False)  # Masquer le subplot en cas d'erreur

# Masquer les subplots vides
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(viz_path, '02_outliers_all_variables.png'))
plt.close()

# Optionnel: Créer un tableau résumé des outliers
outlier_df = pd.DataFrame.from_dict(outlier_stats, orient='index')
print("\nRésumé des outliers:")
print(outlier_df)


Analyse des outliers pour les variables numériques:
DATE_OF_BIRTH: 33788 outliers (2.64%)
CUSTOMER_SINCE: 30 outliers (0.00%)
AGE: 4380 outliers (0.34%)
TENURE: 30 outliers (0.00%)
SOLDE_MOYEN: 29432 outliers (2.30%)
OPERATIONS_MENSUELLES_MOY: 32172 outliers (2.51%)
RATIO_ACTIVITE: 13537 outliers (1.06%)
AGE_A_OUVERTURE: 29034 outliers (2.27%)

Résumé des outliers:
                           count  percentage
DATE_OF_BIRTH              33788    2.640835
CUSTOMER_SINCE                30    0.002345
AGE                         4380    0.342336
TENURE                        30    0.002345
SOLDE_MOYEN                29432    2.300374
OPERATIONS_MENSUELLES_MOY  32172    2.514530
RATIO_ACTIVITE             13537    1.058038
AGE_A_OUVERTURE            29034    2.269267
